# Build the CheckAMG database

## Install required packages

In [1]:
! pip install 'pyhmmer==0.11.1' --quiet

## Global settings

In [2]:
N_THREADS = 16

## General functions

In [3]:
import os
import requests
import shutil
import gzip
import tarfile
from pathlib import Path
from urllib.request import urlretrieve
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from pyhmmer import easel, plan7, hmmer
from __future__ import annotations
from dataclasses import dataclass
from typing import List
import subprocess

In [4]:
def try_download(label, dest, ftp_url, https_url, verify=True):
    if ftp_url:
        try:
            print(f"Trying FTP download for {label}...")
            urlretrieve(ftp_url, dest)
            print(f"{label} downloaded via FTP.")
        except Exception as ftp_err:
            print(f"FTP failed for {label}, falling back to HTTPS: {ftp_err}")
            r = requests.get(https_url, stream=True, verify=verify)
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"{label} downloaded via HTTPS.")
    else:
        try:
            print(f"Trying HTTPS download for {label}...")
            r = requests.get(https_url, stream=True, verify=verify)
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"{label} downloaded via HTTPS.")
        except Exception as e:
            raise RuntimeError(f"Failed to download {label}: {e}")

def hmm_db_complete(dest_path):
    prefix = str(dest_path).replace('.hmm', '')
    required = [f"{prefix}.h3m", f"{prefix}.h3i", f"{prefix}.h3f", f"{prefix}.h3p"]
    return all(Path(f).exists() for f in required)

def fix_hmm_names(file):
    # Fix HMM names to be unique by appending a count suffix
    # Even if the order of the HMMs changes in the source file,
    # this shouldn't affect mapping to descriptions used by CheckAMG,
    # since the 'ACC" field is used for matching, and those are
    # unique in FOAM.
    print(f"Making HMM names unique in {file}")
    unique_counts = defaultdict(int)
    output = []
    with open(file) as infile:
        block = []
        for line in infile:
            if line.startswith("//"):
                # At end of block, fix name
                for i, l in enumerate(block):
                    if l.startswith("NAME"):
                        name = l.strip().split()[1]
                        unique_counts[name] += 1
                        new_name = f"{name}_{unique_counts[name]}"
                        block[i] = f"NAME  {new_name}\n"
                        break
                output.extend(block + [line])
                block = []
            else:
                block.append(line)
    with open(file, "w") as out:
        out.writelines(output)
        
def build_hmm_from_fasta(msa_path: Path, output_path: Path):
    alphabet = easel.Alphabet.amino()
    builder = plan7.Builder(alphabet)
    background = plan7.Background(alphabet)
    with easel.MSAFile(str(msa_path), digital=True, alphabet=alphabet) as msa_file:
        msa = msa_file.read()
        msa.name = msa.accession = msa_path.stem.encode()
        profile, _, _ = builder.build_msa(msa, background)
        with open(output_path, 'wb') as f:
            profile.write(f)

def build_all_phrog_hmms(msa_dir: Path, out_path: Path, threads: int = 10):
    msa_subdirs = [d for d in msa_dir.iterdir() if d.is_dir()]
    if not msa_subdirs:
        raise RuntimeError("No subdirectory with MSA files found in extracted PHROG archive.")
    msa_data_dir = msa_subdirs[0]
    tmp_hmm_dir = msa_dir / "phrog_hmms"
    tmp_hmm_dir.mkdir(parents=True, exist_ok=True)
    print("Building HMMs from PHROG MSAs...")
    with ThreadPoolExecutor(max_workers=threads) as executor:
        futures = [executor.submit(build_hmm_from_fasta, msa_file, tmp_hmm_dir / (msa_file.stem + ".hmm")) for msa_file in msa_data_dir.glob("*.fma")]
        for future in as_completed(futures):
            try:
                future.result()
            except Exception as e:
                print(f"Failed to build HMM: {e}")
    merge_hmm_files_from_dir(tmp_hmm_dir, out_path)
    print(f"Merged PHROG HMMs into {out_path}")
    hmmpress_file(out_path)

def merge_hmm_files_from_dir(src_dir, output_path):
    print(f"Merging HMM files from {src_dir} into {output_path}")
    with open(output_path, 'wb') as out_f:
        for hmm_file in sorted(Path(src_dir).rglob('*.hmm')):
            if hmm_file.is_file() and hmm_file.stat().st_size > 0:
                with open(hmm_file, 'rb') as in_f:
                    shutil.copyfileobj(in_f, out_f)
    shutil.rmtree(src_dir)

def hmmpress_file(hmm_path):
    print(f"Pressing HMM file {hmm_path}")
    hmms = list(plan7.HMMFile(hmm_path))
    output_prefix = str(hmm_path).replace('.hmm', '')
    for ext in ['.h3m', '.h3i', '.h3f', '.h3p']:
        p = Path(f"{output_prefix}{ext}")
        if p.exists():
            p.unlink()
    hmmer.hmmpress(hmms, output_prefix)
    print(f"Pressed HMM database written to {output_prefix}.h3*")

def download_database(label, dest_path, ftp_url, https_url, force=False, decompress=False, untar=False, merge=False, threads=10):
    if hmm_db_complete(dest_path) and not force:
        print(f"{label} already downloaded.")
        return
    tmp_path = dest_path.with_suffix('.tmp')
    if label == "PHROGs":
        try_download(label, tmp_path, ftp_url, https_url, verify=False)
    else:
        try_download(label, tmp_path, ftp_url, https_url, verify=True)
    if untar:
        extract_dir = dest_path.parent / f"{label}_extracted"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tmp_path, 'r:gz') as tar:
            tar.extractall(path=extract_dir)
        tmp_path.unlink()
        print(f"{label} downloaded and extracted to {extract_dir}")
        if label == "PHROGs":
            build_all_phrog_hmms(extract_dir, dest_path, threads=threads)
            shutil.rmtree(extract_dir)
        elif merge:
            merge_hmm_files_from_dir(extract_dir, dest_path)
            hmmpress_file(dest_path)
    elif decompress:
        with gzip.open(tmp_path, 'rb') as f_in, open(dest_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        tmp_path.unlink()
        if label == "FOAM":
            fix_hmm_names(dest_path)
        hmmpress_file(dest_path)
    else:
        tmp_path.rename(dest_path)
        hmmpress_file(dest_path)

def open_text_auto_gz(path):
    with open(path, 'rb') as probe:
        head = probe.read(2)
    if head == b'\x1f\x8b':
        return gzip.open(path, 'rt')
    return open(path, 'r')

def get_thresholds_from_foam(foam_hmm_path, dest_path):
    print(f"Extracting FOAM thresholds from {foam_hmm_path}")
    acc = None
    cutoff_map = {}
    with open(foam_hmm_path, 'r') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith('ACC'):
                parts = line.split()
                if len(parts) >= 2:
                    acc = parts[1].strip(';')
            elif line.startswith('TC') and acc:
                parts = line.replace(';', '').split()
                full = float(parts[1]) if len(parts) > 1 else None
                dom = float(parts[2]) if len(parts) > 2 else None
                cutoff_map[acc] = (full, dom)
                acc = None
    with open(dest_path, 'w') as out:
        out.write('id\tcutoff_full\tcutoff_domain\n')
        for acc, tup in cutoff_map.items():
            full = '' if tup[0] is None else f"{tup[0]}"
            dom = '' if tup[1] is None else f"{tup[1]}"
            out.write(f"{acc}\t{full}\t{dom}\n")
    print(f"FOAM thresholds written to {dest_path}")

def get_thresholds_from_kegg(ftp_url, https_url, dest_path):
    tmp_path = Path(dest_path).with_suffix('.tmp')
    try_download("KEGG thresholds", tmp_path, ftp_url, https_url)
    with open_text_auto_gz(tmp_path) as f_in, open(dest_path, 'w') as f_out:
        f_out.write('id\tthreshold\n')
        for idx, raw in enumerate(f_in):
            line = raw.rstrip('\n')
            if idx == 0:
                cols = [c.strip() for c in line.split('\t')]
                assert len(cols) >= 2, "ko_list header must have at least two columns"
                assert cols[0].lower() == 'knum', f"Expected first column 'knum', got '{cols[0]}'"
                assert cols[1].lower() == 'threshold', f"Expected second column 'threshold', got '{cols[1]}'"
                continue
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if parts and parts[0].startswith('K'):
                knum = parts[0].strip()
                thr = parts[1].strip() if len(parts) > 1 else ''
                if thr == "-":
                    thr = ''
                f_out.write(f"{knum}\t{thr}\n")
    try:
        tmp_path.unlink()
    except Exception:
        pass
    print(f"KEGG thresholds written to {dest_path}")

def get_thresholds_from_camper(ftp_url, https_url, dest_path):
    tmp_path = Path(dest_path).with_suffix('.tmp')
    try_download("CAMPER thresholds", tmp_path, ftp_url, https_url)
    cutoff_map = {}
    with open(tmp_path) as f_in:
        for idx, raw in enumerate(f_in):
            line = raw.rstrip('\n')
            if idx == 0:
                cols = [c.strip() for c in line.split('\t')]
                assert len(cols) >= 4, "CAMPER hmm scores header must have at least four columns"
                assert cols[0].lower() == 'hmm_name', f"Expected first column 'hmm_name', got '{cols[0]}'"
                assert cols[1].lower() == 'a_rank', f"Expected second column 'A_rank', got '{cols[1]}'"
                assert cols[2].lower() == 'b_rank', f"Expected third column 'B_rank', got '{cols[2]}'"
                assert cols[3].lower() == 'score_type', f"Expected fourth column 'score_type', got '{cols[3]}'"
                continue
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            hmm_id = parts[0].strip().replace('.hmm', '') # A few CAMPER HMMs have .hmm suffix in their names
            if not hmm_id:
                continue
            type = parts[3].strip() if len(parts) > 3 else ''
            full, dom = None, None
            if type == "full":
                full = float(parts[1].strip()) if len(parts) > 1 else None
            elif type == "domain":
                dom = float(parts[1].strip()) if len(parts) > 1 else None
            cutoff_map[hmm_id] = (full, dom)
    try:
        tmp_path.unlink()
    except Exception:
        pass
    
    with open(dest_path, 'w') as out:
        out.write('id\tcutoff_full\tcutoff_domain\n')
        for acc, tup in cutoff_map.items():
            full = '' if tup[0] is None else f"{tup[0]}"
            dom = '' if tup[1] is None else f"{tup[1]}"
            out.write(f"{acc}\t{full}\t{dom}\n")
            
    print(f"CAMPER thresholds written to {dest_path}")

@dataclass
class DbEntry:
    database: str
    version: str
    accessed: str
    citation: str

def render_pretty_table(entries: List[DbEntry]) -> str:
    headers = ["database", "version", "accessed", "citation"]
    rows = [[e.database, e.version, e.accessed, e.citation] for e in entries]

    def esc(s: str) -> str:
        # keep a plain-text table stable; avoid tabs/newlines breaking alignment
        return str(s).replace("\t", " ").replace("\n", " ").strip()

    # compute column widths (monospace assumption)
    widths = [len(h) for h in headers]
    for r in rows:
        for i, cell in enumerate(r):
            widths[i] = max(widths[i], len(esc(cell)))

    # build aligned, space-separated table
    header_line = "  ".join(headers[i].ljust(widths[i]) for i in range(len(headers)))
    sep_line = "-" * len(header_line)
    row_lines = [
        "  ".join(esc(r[i]).ljust(widths[i]) for i in range(len(headers)))
        for r in rows
    ]
    return "\n".join([header_line, sep_line] + row_lines)

def build_readme(
    entries: List[DbEntry],
    checkamg_db_version: str,
    checkamg_db_date: str,
) -> str:
    table = render_pretty_table(entries)
    return (
        f"CheckAMG database version {checkamg_db_version} ({checkamg_db_date})\n"
        "\n"
        "The CheckAMG database is a collection of pre-existing databases of protein families and profile HMMs.\n"
        "Please cite the original publications for these databases when using CheckAMG in your research.\n"
        "\n"
        "All databases were downloaded as profile HMMs and reformatted as needed for compatibility.\n"
        "The exception is PHROGs which was downloaded as MSA fastas and built into HMM profiles.\n"
        "See https://github.com/AnantharamanLab/CheckAMG/blob/main/notebooks/build_checkamg_db.ipynb\n"
        "for details.\n"
        "\n"
        f"{table}\n"
    )

def make_tar_gz_parallel(src_dir: str, threads: int = 0, out_path: str | None = None) -> Path:
    src = Path(src_dir).resolve()
    if not src.is_dir():
        raise FileNotFoundError(f"Source directory not found: {src}")

    if shutil.which("tar") is None:
        raise RuntimeError("tar not found on PATH")
    if shutil.which("pigz") is None:
        raise RuntimeError("pigz not found on PATH")

    out = Path(out_path).resolve() if out_path else src.parent / f"{src.name}.tar.gz"
    out.parent.mkdir(parents=True, exist_ok=True)

    # threads=0 -> pigz default (uses available cores)
    cmd = [
        "tar", "-C", str(src.parent), "-cf", "-", src.name,
    ]
    pigz = ["pigz", "-9"]
    if threads and threads > 0:
        pigz += ["-p", str(threads)]
    else:
        pigz += ["-p", "0"]

    with open(out, "wb") as f_out:
        p1 = subprocess.Popen(cmd, stdout=subprocess.PIPE)
        try:
            p2 = subprocess.Popen(pigz, stdin=p1.stdout, stdout=f_out)
        finally:
            if p1.stdout is not None:
                p1.stdout.close()

        rc2 = p2.wait()
        rc1 = p1.wait()

    if rc1 != 0:
        raise RuntimeError(f"tar failed with exit code {rc1}")
    if rc2 != 0:
        raise RuntimeError(f"pigz failed with exit code {rc2}")

    return out

## v1.0 (2025-12-15)
Compatible with **CheckAMG v0.7.0** and higher.

| Database   | Version    | Accessed   | Citation                                                                                                                                 |
| ---------- | ---------- | ---------- | ---------------------------------------------------------------------------------------------------------------------------------------- |
| KEGG KOfam | 2025-12-01 | 2025-12-15 | Aramaki et al. 2020, *Bioinformatics*. [https://doi.org/10.1093/bioinformatics/btz859](https://doi.org/10.1093/bioinformatics/btz859)    |
| Pfam-A     | 38         | 2025-12-15 | Mistry et al. 2021, *Nucleic Acids Research*. [https://doi.org/10.1093/nar/gkaa913](https://doi.org/10.1093/nar/gkaa913)                 |
| FOAM       | 2023-01-10 | 2025-12-15 | Prestat et al. 2014, *Nucleic Acids Research*. [https://doi.org/10.1093/nar/gku702](https://doi.org/10.1093/nar/gku702)                  |
| PHROGs     | 4          | 2025-12-15 | Terzian et al. 2021, *NAR Genomics and Bioinformatics*. [https://doi.org/10.1093/nargab/lqab067](https://doi.org/10.1093/nargab/lqab067) |
| dbCAN      | 14         | 2025-12-15 | Drula et al. 2022, *Nucleic Acids Research*. [https://doi.org/10.1093/nar/gkab1045](https://doi.org/10.1093/nar/gkab1045)                |
| METABOLIC  | 2023-06-12 | 2025-12-15 | Zhou et al. 2022, *Microbiome*. [https://doi.org/10.1186/s40168-021-01213-8](https://doi.org/10.1186/s40168-021-01213-8)                 |
| CAMPER     | 1.0.0      | 2025-12-15 | McGivern et al. 2024. [https://doi.org/10.1101/2023.09.24.559193](https://doi.org/10.1101/2023.09.24.559193)                             |


### Links to database downloads
**NOTES:**
* The KEGG download website structure hosts the latest version of KOfam under `./kofam/profiles.tar.gz`, and older versions under `./kofam/archives/[VERSION]`. At the time of writing this notebook, KOfam version 2025-12-01 was the latest release. But in the future, it may be found under `./kofam/archives/2025-12-01`. For reproducibility, ensure that the correct URL for KOfam version 2025-12-01 is used below.

* PHROGs does not host version-specific URLs for its downloads. The only available one will be used below, which as of Jan 28 2026 corresponded to PHROGs version 4 (June 15 2022).

* The FOAM and METABOLIC HMM download links are similarly not linked to specific versions, but new official releases have not been made.

* The link to CAMPER HMMs also is not version specific. New commits to the repository that introduce an updated set of profile HMMs may lead to a different database than the one built here.

In [5]:
KEGG_FTP = 'ftp://ftp.genome.jp/pub/db/kofam/profiles.tar.gz'
KEGG_HTTPS = 'https://www.genome.jp/ftp/db/kofam/profiles.tar.gz'

PFAM_FTP = 'ftp://ftp.ebi.ac.uk/pub/databases/Pfam/releases/Pfam38.0/Pfam-A.hmm.gz'
PFAM_HTTPS = 'https://ftp.ebi.ac.uk/pub/databases/Pfam/releases/Pfam38.0/Pfam-A.hmm.gz'

FOAM_FTP = None
FOAM_HTTPS = 'https://osf.io/download/bdpv5'

PHROGS_FTP = None
PHROGS_HTTPS = 'https://phrogs.lmge.uca.fr/downloads_from_website/MSA_phrogs.tar.gz'

DBCAN_FTP = None
DBCAN_HTTPS = 'http://dbcan-hcc.unl.edu/download/dbCAN-HMMdb-V14.txt'

METABOLIC_FTP = None
METABOLIC_HTTPS = 'https://github.com/AnantharamanLab/CheckAMG/raw/refs/heads/main/custom_dbs/METABOLIC_custom.hmm.gz'

CAMPER_FTP = None
CAMPER_HTTPS = 'https://raw.githubusercontent.com/WrightonLabCSU/CAMPER/refs/heads/main/CAMPER.hmm'
CAMPER_SCORES_FTP = None
CAMPER_SCORES_HTTPS = 'https://raw.githubusercontent.com/WrightonLabCSU/CAMPER/refs/heads/main/CAMPER_hmm_scores.tsv'

### Download, format, and build database files

In [6]:
CHECKAMG_DB_VERSION = 1.0
CHECKAMG_DB_DATE = "2026-01-28"

In [7]:
dest = Path(f"./CheckAMG_db_v{str(CHECKAMG_DB_VERSION).replace('.0', '')}_{CHECKAMG_DB_DATE.replace('-', '')}")
print(f"Output destination: {dest}")
os.makedirs(dest, exist_ok=True)

Output destination: CheckAMG_db_v1_20260128


In [8]:
# Force redownload of all databases
force = True

In [9]:
print("Starting download of all databases.")
dbs = [
    ("KEGG", 'KEGG.hmm', KEGG_FTP, KEGG_HTTPS, True, True, True),
    ("Pfam", 'Pfam-A.hmm', PFAM_FTP, PFAM_HTTPS, True, False, False),
    ("FOAM", 'FOAM.hmm', FOAM_FTP, FOAM_HTTPS, True, False, False),
    ("PHROGs", 'PHROGs.hmm', PHROGS_FTP, PHROGS_HTTPS, False, True, True),
    ("dbCAN", 'dbCAN_HMMdb_v14.hmm', DBCAN_FTP, DBCAN_HTTPS, False, False, False),
    ("METABOLIC", 'METABOLIC_custom.hmm', METABOLIC_FTP, METABOLIC_HTTPS, True, False, False),
    ("CAMPER", 'CAMPER.hmm', CAMPER_FTP, CAMPER_HTTPS, False, False, False),
]
exceptions = []
with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
    futures = {
        executor.submit(download_database, name, Path(dest)/fname, ftp, https, force, decomp, untar, merge, N_THREADS): name
        for name, fname, ftp, https, decomp, untar, merge in dbs
    }
    for future in as_completed(futures):
        name = futures[future]
        try:
            future.result()
        except Exception as e:
            print(f"Error downloading {name}: {e}")
            exceptions.append(name)
if exceptions:
    raise Exception(f"Download failed for: {', '.join(exceptions)}")
else:
    print("All databases downloaded successfully.")

Starting download of all databases.
Trying FTP download for KEGG...
Trying FTP download for Pfam...
Trying HTTPS download for FOAM...
Trying HTTPS download for PHROGs...
Trying HTTPS download for dbCAN...
Trying HTTPS download for METABOLIC...
Trying HTTPS download for CAMPER...


/storage2/scratch/kosmopoulos/software/CheckAMG/.conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'phrogs.lmge.uca.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


METABOLIC downloaded via HTTPS.
Pressing HMM file CheckAMG_db_v1_20260128/METABOLIC_custom.hmm
Pressed HMM database written to CheckAMG_db_v1_20260128/METABOLIC_custom.h3*
CAMPER downloaded via HTTPS.
Pressing HMM file CheckAMG_db_v1_20260128/CAMPER.hmm
Pressed HMM database written to CheckAMG_db_v1_20260128/CAMPER.h3*
PHROGs downloaded via HTTPS.
dbCAN downloaded via HTTPS.
Pressing HMM file CheckAMG_db_v1_20260128/dbCAN_HMMdb_v14.hmm
Pressed HMM database written to CheckAMG_db_v1_20260128/dbCAN_HMMdb_v14.h3*
PHROGs downloaded and extracted to CheckAMG_db_v1_20260128/PHROGs_extracted
Building HMMs from PHROG MSAs...
FOAM downloaded via HTTPS.
Making HMM names unique in CheckAMG_db_v1_20260128/FOAM.hmm
Pfam downloaded via FTP.
Pressing HMM file CheckAMG_db_v1_20260128/FOAM.hmm
Pressing HMM file CheckAMG_db_v1_20260128/Pfam-A.hmm
Pressed HMM database written to CheckAMG_db_v1_20260128/Pfam-A.h3*
Merging HMM files from CheckAMG_db_v1_20260128/PHROGs_extracted/phrog_hmms into CheckAMG_db_

### Build threshold files for FOAM, KEGG, and CAMPER

In [10]:
try:
    foam_hmm = Path(dest) / 'FOAM.hmm'
    foam_thr = Path(dest) / 'FOAM_cutoffs.tsv'
    if foam_hmm.exists():
        get_thresholds_from_foam(foam_hmm, foam_thr)
    else:
        print("FOAM.hmm not found; skipping FOAM thresholds.")
except Exception as e:
    print(f"FOAM thresholds failed: {e}")
    exceptions.append("FOAM_thresholds")

Extracting FOAM thresholds from CheckAMG_db_v1_20260128/FOAM.hmm
FOAM thresholds written to CheckAMG_db_v1_20260128/FOAM_cutoffs.tsv


In [11]:
try:
    KO_LIST_FTP = KEGG_FTP.replace('profiles.tar.gz', 'ko_list.gz')
    KO_LIST_HTTPS = KEGG_HTTPS.replace('profiles.tar.gz', 'ko_list.gz')
    kegg_thr = Path(dest) / 'KEGG_cutoffs.tsv'
    get_thresholds_from_kegg(KO_LIST_FTP, KO_LIST_HTTPS, kegg_thr)
except Exception as e:
    print(f"KEGG thresholds failed: {e}")
    exceptions.append("KEGG_thresholds")

Trying FTP download for KEGG thresholds...
KEGG thresholds downloaded via FTP.
KEGG thresholds written to CheckAMG_db_v1_20260128/KEGG_cutoffs.tsv


In [12]:
try:
    camper_hmm = Path(dest) / 'CAMPER.hmm'
    camper_thr = Path(dest) / 'CAMPER_cutoffs.tsv'
    if camper_hmm.exists():
        get_thresholds_from_camper(CAMPER_SCORES_FTP, CAMPER_SCORES_HTTPS, camper_thr)
    else:
        print("CAMPER.hmm not found; skipping CAMPER thresholds.")
except Exception as e:
    print(f"CAMPER thresholds failed: {e}")
    exceptions.append("CAMPER_thresholds")

Trying HTTPS download for CAMPER thresholds...
CAMPER thresholds downloaded via HTTPS.
CAMPER thresholds written to CheckAMG_db_v1_20260128/CAMPER_cutoffs.tsv


In [13]:
if exceptions:
    raise Exception(f"Completed with errors in: {', '.join(exceptions)}")
else:
    print("All database thresholds prepared successfully.")

All database thresholds prepared successfully.


### Package the database

In [19]:
entries = [
    DbEntry(
        database="KEGG KOfam",
        version="2025-12-01",
        accessed="2025-12-15",
        citation="Aramaki et al. 2020 Bioinformatics https://doi.org/10.1093/bioinformatics/btz859",
    ),
    DbEntry(
        database="Pfam-A",
        version="38",
        accessed="2025-12-15",
        citation="Mistry et al. 2021 Nucleic Acids Research https://doi.org/10.1093/nar/gkaa913",
    ),
    DbEntry(
        database="FOAM",
        version="2023-01-10",
        accessed="2025-12-15",
        citation="Prestat et al. 2014 Nucleic Acids Research https://doi.org/10.1093/nar/gku702",
    ),
    DbEntry(
        database="PHROGs",
        version="4",
        accessed="2025-12-15",
        citation="Terzian et al. 2021 NAR Genomics and Bioinformatics https://doi.org/10.1093/nargab/lqab067",
    ),
    DbEntry(
        database="dbCAN",
        version="14",
        accessed="2025-12-15",
        citation="Drula et al. 2022 Nucleic Acids Research https://doi.org/10.1093/nar/gkab1045",
    ),
    DbEntry(
        database="METABOLIC",
        version="2023-06-12",
        accessed="2025-12-15",
        citation="Zhou et al. 2022 Microbiome https://doi.org/10.1186/s40168-021-01213-8",
    ),
    DbEntry(
        database="CAMPER",
        version="1.0.0",
        accessed="2025-12-15",
        citation="McGivern et al. 2024 https://doi.org/10.1101/2023.09.24.559193",
    ),
]

In [20]:
readme = build_readme(entries, CHECKAMG_DB_VERSION, CHECKAMG_DB_DATE)

In [21]:
print(readme)

CheckAMG database version 1.0 (2026-01-28)

The CheckAMG database is a collection of pre-existing databases of protein families and profile HMMs.
Please cite the original publications for these databases when using CheckAMG in your research.

All databases were downloaded as profile HMMs and reformatted as needed for compatibility.
The exception is PHROGs which was downloaded as MSA fastas and built into HMM profiles.
See https://github.com/AnantharamanLab/CheckAMG/blob/main/notebooks/build_checkamg_db.ipynb
for details.

database    version     accessed    citation                                                                                  
------------------------------------------------------------------------------------------------------------------------------
KEGG KOfam  2025-12-01  2025-12-15  Aramaki et al. 2020 Bioinformatics https://doi.org/10.1093/bioinformatics/btz859          
Pfam-A      38          2025-12-15  Mistry et al. 2021 Nucleic Acids Research https://doi.o

In [22]:
with open(dest / "README.txt", 'w') as f:
    f.write(readme)

In [23]:
packaged = make_tar_gz_parallel(dest, threads=32)
print(f"Wrote: {packaged}")

Wrote: /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_db_v1_20260128.tar.gz
